In [ ]:
!git clone -b toy_transformer --single-branch https://github.com/Jayden3316/Viveka.git


In [ ]:
import os
os.chdir("/kaggle/working/Viveka/toy_transformer/ICL")


In [ ]:
!pip install transformer-lens==2.16.1 \
             transformers==4.56.1 \
             accelerate==1.10.1 \
             einops==0.8.1 \
             jaxtyping==0.3.2 \
             fancy-einsum==0.0.3 \
             wandb==0.21.3 \
             datasets sentencepiece huggingface-hub


In [ ]:
from toy_model import *
from metrics import *
import wandb
import torch
import numpy as np

In [ ]:
wandb.login(key="9df77e7cbad36f3323af2ea208aa4027a970df97")

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

# Define Markov transition matrices (same as original)
T0_proc1 = np.array([
    [0, 1, 0],
    [0, 0, 1],
    [0, 0, 0.5]
])

T1_proc1 = np.array([
    [0, 0, 0],
    [0, 0, 0],
    [0.5, 0, 0]
])

eta_0_proc1 = np.array([0.25,0.25,0.5],dtype=float)

T0_proc2 = np.array([
    [0,1,0],
    [0,0,0],
    [0.5,0,0]
])
T1_proc2 = np.array([
    [0,0,0],
    [0,0,1],
    [0.5,0,0]
])
T0_proc3 = np.array([
    [0,1,0,0],
    [0,0,0,0.5],
    [0.5,0,0,0],
    [0,0,0,0.5]])
T1_proc3 = np.array([
    [0,0,0,0],
    [0,0,0.5,0],
    [0.5,0,0,0],
    [0.5,0,0,0]]) 
eta_0_proc3=np.array([2/7,2/7,1/7,2/7],dtype=float)

# Create device-aware processes
process1 = MarkovData(n_gen=100, gen_len=32, n_states=3, d_vocab=2, 
                     T_list=[T0_proc1, T1_proc1], eta0=eta_0_proc1, seed=43, device=device)
process2 = MarkovData(n_gen=100, gen_len=32, n_states=3, d_vocab=2, 
                     T_list=[T0_proc2, T1_proc2], seed=43, device=device)
process3 = MarkovData(n_gen=100, gen_len=32, n_states=4, d_vocab=2, 
                     T_list=[T0_proc3, T1_proc3], eta0=eta_0_proc3, seed=43, device=device)

In [ ]:
dataset1 = MarkovData(n_gen=50000, gen_len=32, n_states=3, d_vocab=2, 
                     T_list=[T0_proc1, T1_proc1], eta0=eta_0_proc1, device=device)
dataset2 = MarkovData(n_gen=50000, gen_len=32, n_states=3, d_vocab=2, 
                     T_list=[T0_proc2, T1_proc2], device=device)
merged_dataset = MergeMarkovDatasets(dataset1=dataset1, dataset2=dataset2, 
                                   mixing_style='random')

In [ ]:
dataset1_test = MarkovData(n_gen=50, gen_len=32, n_states=3, d_vocab=2, 
                          T_list=[T0_proc1, T1_proc1], eta0=eta_0_proc1, seed=43, device=device)
dataset2_test = MarkovData(n_gen=50, gen_len=32, n_states=3, d_vocab=2, 
                          T_list=[T0_proc2, T1_proc2], seed=43, device=device)

test_dataset_merged = MergeMarkovDatasets(dataset1=dataset1_test, dataset2=dataset2_test, 
                                        mixing_style='random')

In [ ]:

metrics_config = MetricsConfig(

    track_markov_kl=True,
    markov_processes=[process1, process2, process3], 
    pos_start=3,
    
    track_ngrams=True,
    ngram_data=test_dataset_merged,
    ngram_orders=[1, 2, 3],

    track_previous_token=True,
    prev_token_data=test_dataset_merged,
    track_in_context=True, 
    icl_data=test_dataset_merged,
    icl_k1=5,
    icl_k2=30,
    track_composition=False,
    #composition_layers=[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)],
    track_prefix_matching=False,
    device=device
)


In [ ]:
model = train_model(
    dataset=merged_dataset,
    
    # Architecture
    n_layers=4,
    d_model=16,
    n_heads=4,
    attn_only=False,
    act_fn='silu',
    normalization_type='LN',
    d_mlp=64,

    # Training hyperparameters
    n_epochs=300,
    batch_size=64,
    lr=0.05,
    device=device,  # Explicit device specification

    # Logging and saving
    wandb=True,
    wandb_project_name="ICL",
    save_dir="seq_len32/x9*_ln",
    save_every=5,
    print_every=10,

    # Comprehensive metrics tracking
    metrics_config=metrics_config,
    metrics_log_interval=20
)

print(f"✅ Training completed! Model device: {next(model.parameters()).device}")
wandb.finish()

In [ ]:
!zip -r current_run_checkpoints.zip seq_len32/x9*_ln
